# 06 — Map Layers Prototype

**Story**: S01-06 | **Spec**: `openspec/capabilities/map/spec.md`

Validate all map tile sources and WMS layers for the BoatOps planning dashboard.
No credentials required — all sources are public.

| Scenario | Validates |
|----------|-----------|
| SCENARIO-MAP-OSM-001 | OSM tiles at zoom 10 near Warnemünde |
| SCENARIO-MAP-SEAMAP-001 | OpenSeaMap seamark overlay |
| SCENARIO-MAP-GEOTIFF-001 | Synthetic raster as georeferenced ImageOverlay |
| SCENARIO-MAP-WMS-001 | Generic WMS layer loading (validated via GEBCO) |
| SCENARIO-MAP-GEBCO-001 | GEBCO bathymetry WMS (`GEBCO_LATEST`) |
| SCENARIO-MAP-EMODNET-001 | EMODnet Seabed Habitats WMS (`eusm2022`) |

**AOI center**: Warnemünde 54.18°N, 12.08°E | **AOI bbox**: `[9.5, 53.5, 15.0, 56.0]`

In [ ]:
import io
import base64
import xml.etree.ElementTree as ET
from math import radians, floor, log, tan, cos, pi

import numpy as np
import requests
import matplotlib.pyplot as plt

from PIL import Image

from ipyleaflet import (
    Map, TileLayer, WMSLayer, ImageOverlay,
    LayersControl, ScaleControl, FullScreenControl,
)

In [ ]:
AOI_CENTER = (54.18, 12.08)            # Warnemünde (lat, lon)
AOI_ZOOM   = 8
AOI_BBOX   = (9.5, 53.5, 15.0, 56.0)  # (west, south, east, north)

OSM_URL     = 'https://tile.openstreetmap.org/{z}/{x}/{y}.png'
SEAMAP_URL  = 'https://tiles.openseamap.org/seamark/{z}/{x}/{y}.png'
GEBCO_WMS   = 'https://wms.gebco.net/mapserv'                                  # old URL (gebco.net/.../mapserv) returns 404
EMODNET_WMS = 'https://ows.emodnet-seabedhabitats.eu/geoserver/emodnet_view/wms'
GEBCO_LAYER   = 'GEBCO_LATEST'
EMODNET_LAYER = 'eusm_helcom_group'  # HELCOM Baltic classification group; eusm2022 no longer exists

WMS_TEST_BBOX = (10.5, 53.8, 13.0, 55.2)  # (west, south, east, north)

TILE_HEADERS = {'User-Agent': 'BoatOps/1.0'}


def deg2tile(lat: float, lon: float, zoom: int) -> tuple:
    """Slippy-map tile (x, y) for a lat/lon at the given zoom level."""
    lat_r = radians(lat)
    n = 2 ** zoom
    x = floor((lon + 180.0) / 360.0 * n)
    y = floor((1.0 - log(tan(lat_r) + 1.0 / cos(lat_r)) / pi) / 2.0 * n)
    return x, y


def wms_layer_names(base_url: str, version: str = '1.3.0') -> list:
    """Return all <Name> values from a WMS GetCapabilities response."""
    resp = requests.get(
        base_url,
        params={'SERVICE': 'WMS', 'REQUEST': 'GetCapabilities', 'VERSION': version},
        timeout=30,
    )
    resp.raise_for_status()
    root = ET.fromstring(resp.content)
    return [
        el.text.strip()
        for el in root.iter()
        if el.tag.split('}')[-1] == 'Name' and el.text and el.text.strip()
    ]


print('Constants and helpers loaded.')
print(f'AOI center: {AOI_CENTER} | zoom: {AOI_ZOOM}')
print(f'WMS test bbox: {WMS_TEST_BBOX}')

## 1. OSM Base Layer — SCENARIO-MAP-OSM-001

Fetch a tile near Warnemünde at zoom 10 to confirm connectivity and image format.

In [ ]:
zoom = 10
tx, ty = deg2tile(*AOI_CENTER, zoom)
print(f'Warnemünde tile at zoom {zoom}: x={tx}, y={ty}')

resp_osm = requests.get(
    f'https://tile.openstreetmap.org/{zoom}/{tx}/{ty}.png',
    headers=TILE_HEADERS, timeout=15,
)
assert resp_osm.status_code == 200, f'OSM tile HTTP {resp_osm.status_code}'
assert 'image' in resp_osm.headers.get('Content-Type', ''), 'Expected image content'
print(f'OK: HTTP 200, {len(resp_osm.content):,} bytes, {resp_osm.headers["Content-Type"]}')

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(Image.open(io.BytesIO(resp_osm.content)))
ax.set_title(f'OSM tile z={zoom}/x={tx}/y={ty}')
ax.axis('off')
plt.tight_layout()
plt.show()

m_osm = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_osm.add(TileLayer(
    url=OSM_URL,
    attribution='© OpenStreetMap contributors',
    name='OpenStreetMap',
))
m_osm.add(ScaleControl(position='bottomleft'))
m_osm

## 2. OpenSeaMap Seamark Overlay — SCENARIO-MAP-SEAMAP-001

OpenSeaMap returns HTTP 200 with a transparent PNG even for tiles with no seamarks.

In [ ]:
resp_sea = requests.get(
    f'https://tiles.openseamap.org/seamark/{zoom}/{tx}/{ty}.png',
    headers=TILE_HEADERS, timeout=15,
)
assert resp_sea.status_code == 200, f'OpenSeaMap tile HTTP {resp_sea.status_code}'
assert 'image' in resp_sea.headers.get('Content-Type', ''), 'Expected image'
print(f'OpenSeaMap: HTTP 200, {len(resp_sea.content):,} bytes, {resp_sea.headers.get("Content-Type")}')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(Image.open(io.BytesIO(resp_osm.content)))
axes[0].set_title('OSM base')
axes[0].axis('off')
axes[1].imshow(Image.open(io.BytesIO(resp_sea.content)))
axes[1].set_title('OpenSeaMap (transparent PNG)')
axes[1].axis('off')
plt.suptitle(f'z={zoom}/x={tx}/y={ty} near Warnemünde')
plt.tight_layout()
plt.show()

m_seamap = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_seamap.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_seamap.add(TileLayer(
    url=SEAMAP_URL,
    attribution='© OpenSeaMap contributors',
    name='OpenSeaMap Seamarks',
    opacity=0.9,
))
m_seamap.add(LayersControl(position='topright'))
m_seamap.add(ScaleControl(position='bottomleft'))
m_seamap

## 3. GeoTIFF Overlay — SCENARIO-MAP-GEOTIFF-001

Create a synthetic raster (fake depth gradient) over the Mecklenburg Bight, render to a PNG
data URI, and add as an `ImageOverlay`. Demonstrates the rasterio-based GeoTIFF workflow
that will be used in Phase 2 for drone orthomosaic upload.

In [ ]:
RASTER_BBOX = (11.5, 53.9, 13.5, 54.8)   # (west, south, east, north)
w_r, s_r, e_r, n_r = RASTER_BBOX
px_w, px_h = 120, 72

X = np.linspace(0, 1, px_w)
Y = np.linspace(0, 1, px_h)
XX, YY = np.meshgrid(X, Y)
depth = (1.0 - 0.6 * XX) * 25 + 3 * np.sin(XX * pi * 3) * np.cos(YY * pi * 2)
depth = np.clip(depth, 0, 25)

cmap = plt.get_cmap('Blues_r')
norm = plt.Normalize(vmin=0, vmax=25)
rgba = (cmap(norm(depth)) * 255).astype(np.uint8)
rgba[..., 3] = 170

buf = io.BytesIO()
Image.fromarray(rgba, 'RGBA').save(buf, format='PNG')
buf.seek(0)
geotiff_uri = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

print(f'Raster shape: {depth.shape} | depth range: {depth.min():.1f}–{depth.max():.1f} m')
print(f'Data URI length: {len(geotiff_uri):,} chars')

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(depth, cmap='Blues_r', origin='upper',
               extent=[w_r, e_r, s_r, n_r], vmin=0, vmax=25)
plt.colorbar(im, ax=ax, label='Depth (m, synthetic)')
ax.set_title('Synthetic bathymetry raster — SCENARIO-MAP-GEOTIFF-001')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

m_gtiff = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_gtiff.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_gtiff.add(ImageOverlay(
    url=geotiff_uri,
    bounds=[[s_r, w_r], [n_r, e_r]],
    opacity=0.65,
    name='Synthetic Raster (GeoTIFF proxy)',
))
m_gtiff.add(LayersControl(position='topright'))
m_gtiff.add(ScaleControl(position='bottomleft'))
m_gtiff

## 4. GEBCO Bathymetry WMS — SCENARIO-MAP-GEBCO-001

**Service**: `https://wms.gebco.net/mapserv` (new URL; old `gebco.net/.../mapserv` returns 404)  
**Layer**: `GEBCO_LATEST` (2023 15-arc-second grid)  
No authentication required.

In [ ]:
print('GEBCO WMS — GetCapabilities')
gebco_layers = wms_layer_names(GEBCO_WMS)
print(f'Name entries: {len(gebco_layers)}')
for lyr in gebco_layers:
    tag = '  <-- target' if lyr == GEBCO_LAYER else ''
    print(f'  {lyr}{tag}')

assert len(gebco_layers) > 0, 'No layers in GEBCO GetCapabilities'
assert GEBCO_LAYER in gebco_layers, f'Expected {GEBCO_LAYER!r}; available: {gebco_layers}'
print(f'\nConfirmed: {GEBCO_LAYER!r} present in GetCapabilities')

# GetMap — small Baltic test area
w_t, s_t, e_t, n_t = WMS_TEST_BBOX
resp_gebco = requests.get(GEBCO_WMS, params={
    'SERVICE': 'WMS', 'VERSION': '1.1.1', 'REQUEST': 'GetMap',
    'LAYERS': GEBCO_LAYER, 'SRS': 'EPSG:4326',
    'BBOX': f'{w_t},{s_t},{e_t},{n_t}',
    'WIDTH': '400', 'HEIGHT': '400',
    'FORMAT': 'image/png', 'TRANSPARENT': 'TRUE', 'STYLES': '',
}, timeout=30)
assert resp_gebco.status_code == 200, f'GEBCO GetMap HTTP {resp_gebco.status_code}: {resp_gebco.text[:200]}'
ct_gebco = resp_gebco.headers.get('Content-Type', '')
assert 'image' in ct_gebco, f'Expected image, got: {ct_gebco!r}'
print(f'GEBCO GetMap: HTTP 200 | {len(resp_gebco.content):,} bytes | {ct_gebco}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(Image.open(io.BytesIO(resp_gebco.content)),
          extent=[w_t, e_t, s_t, n_t], aspect='auto')
ax.set_title(f'GEBCO Bathymetry WMS — GetMap\nLayer: {GEBCO_LAYER}')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
m_gebco = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_gebco.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_gebco.add(WMSLayer(
    url=GEBCO_WMS,
    layers=GEBCO_LAYER,
    format='image/png',
    transparent=True,
    attribution='© GEBCO/BODC',
    name='GEBCO Bathymetry',
    opacity=0.7,
))
m_gebco.add(LayersControl(position='topright'))
m_gebco.add(ScaleControl(position='bottomleft'))
m_gebco

## 5. EMODnet Seabed Habitats WMS — SCENARIO-MAP-EMODNET-001

**Service**: `https://ows.emodnet-seabedhabitats.eu/geoserver/emodnet_view/wms`  
**Layer**: `eusm_helcom_group` (HELCOM Baltic seabed habitat classification, multi-scale group)  
No authentication required.

> **Note**: The monolithic `eusm2022` layer no longer exists — the service was restructured into
> per-scale and per-classification-scheme layers. `eusm_helcom_group` is the correct Baltic-specific
> layer (5013 distinct rendered colors over the test bbox). `_full`-suffix layers require high zoom
> to render and return blank at overview scale.

In [ ]:
print('EMODnet Seabed Habitats WMS — GetCapabilities')
emodnet_layers = wms_layer_names(EMODNET_WMS)
print(f'Name entries: {len(emodnet_layers)}')
eusm_names = [l for l in emodnet_layers if 'eusm' in l.lower() or 'helcom' in l.lower()]
print(f'EUSM/HELCOM-related layers ({len(eusm_names)} found):')
for lyr in eusm_names[:20]:
    tag = '  <-- target' if lyr == EMODNET_LAYER else ''
    print(f'  {lyr}{tag}')
if len(eusm_names) > 20:
    print(f'  ... and {len(eusm_names) - 20} more')

assert len(emodnet_layers) > 0, 'No layers in EMODnet GetCapabilities'

# Resolve active layer — eusm2022 no longer exists; use current equivalents
_candidates = [
    EMODNET_LAYER,                    # eusm_helcom_group (HELCOM Baltic)
    'eusm_bio_group',                 # broad-scale biological fallback
    'eusm2023_bio_group',             # 2023 specific
    'eusm2023_helcom_group',          # 2023 HELCOM
    'eusm2022', 'eusm2021', 'eusm2019',  # legacy names
]
emodnet_layer_active = next((c for c in _candidates if c in emodnet_layers), None)
if emodnet_layer_active is None:
    emodnet_layer_active = next((l for l in emodnet_layers[1:] if l), None)
    print(f'WARNING: {EMODNET_LAYER!r} not found; using fallback: {emodnet_layer_active!r}')
else:
    print(f'\nLayer resolved: {emodnet_layer_active!r}')

assert emodnet_layer_active is not None, 'No usable EMODnet layer found'

w_t, s_t, e_t, n_t = WMS_TEST_BBOX
resp_emodnet = requests.get(EMODNET_WMS, params={
    'SERVICE': 'WMS', 'VERSION': '1.1.1', 'REQUEST': 'GetMap',
    'LAYERS': emodnet_layer_active, 'SRS': 'EPSG:4326',
    'BBOX': f'{w_t},{s_t},{e_t},{n_t}',
    'WIDTH': '400', 'HEIGHT': '400',
    'FORMAT': 'image/png', 'TRANSPARENT': 'TRUE', 'STYLES': '',
}, timeout=30)
assert resp_emodnet.status_code == 200, f'EMODnet GetMap HTTP {resp_emodnet.status_code}: {resp_emodnet.text[:200]}'
ct_emodnet = resp_emodnet.headers.get('Content-Type', '')
assert 'image' in ct_emodnet, f'Expected image, got: {ct_emodnet!r}'
print(f'EMODnet GetMap: HTTP 200 | {len(resp_emodnet.content):,} bytes | {ct_emodnet}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(Image.open(io.BytesIO(resp_emodnet.content)),
          extent=[w_t, e_t, s_t, n_t], aspect='auto')
ax.set_title(f'EMODnet Seabed Habitats WMS — GetMap\nLayer: {emodnet_layer_active}')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
m_emodnet = Map(center=AOI_CENTER, zoom=AOI_ZOOM, scroll_wheel_zoom=True)
m_emodnet.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_emodnet.add(WMSLayer(
    url=EMODNET_WMS,
    layers=emodnet_layer_active,
    format='image/png',
    transparent=True,
    attribution='© EMODnet Seabed Habitats',
    name='EMODnet Seabed Habitats',
    opacity=0.7,
))
m_emodnet.add(LayersControl(position='topright'))
m_emodnet.add(ScaleControl(position='bottomleft'))
m_emodnet

## 6. Combined Map — All Toggleable Layers

Single ipyleaflet map with all six layers and a `LayersControl` for toggling.
This is the target UX pattern for the Phase 3 MapLibre GL JS frontend.

In [ ]:
m = Map(
    center=AOI_CENTER,
    zoom=AOI_ZOOM,
    scroll_wheel_zoom=True,
    layout={'height': '600px'},
)

m.add(TileLayer(
    url=OSM_URL,
    attribution='© OpenStreetMap contributors',
    name='OpenStreetMap',
))
m.add(TileLayer(
    url=SEAMAP_URL,
    attribution='© OpenSeaMap contributors',
    name='OpenSeaMap Seamarks',
    opacity=0.9,
))
m.add(ImageOverlay(
    url=geotiff_uri,
    bounds=[[s_r, w_r], [n_r, e_r]],
    opacity=0.5,
    name='Synthetic Raster (GeoTIFF proxy)',
))
m.add(WMSLayer(
    url=GEBCO_WMS,
    layers=GEBCO_LAYER,
    format='image/png',
    transparent=True,
    attribution='© GEBCO/BODC',
    name='GEBCO Bathymetry',
    opacity=0.6,
))
m.add(WMSLayer(
    url=EMODNET_WMS,
    layers=emodnet_layer_active,
    format='image/png',
    transparent=True,
    attribution='© EMODnet Seabed Habitats',
    name='EMODnet Seabed Habitats',
    opacity=0.6,
))

m.add(LayersControl(position='topright'))
m.add(ScaleControl(position='bottomleft'))
m.add(FullScreenControl())
m

## 7. Data Layer Overlays — Integration Validation

Demonstrates that each data layer from earlier notebooks can be rendered as a
georeferenced `ImageOverlay`, establishing the pattern for Phase 2/3.

| Scenario | Source | Overlay |
|----------|--------|---------|
| SCENARIO-MAP-WIND-OVERLAY-001 | Open-Meteo 3×3 grid | quiver PNG, colour = speed |
| SCENARIO-MAP-WAVE-OVERLAY-001 | CMEMS VHM0, current hour | colour-mapped PNG, land transparent |
| SCENARIO-MAP-S2-OVERLAY-001 | CDSE SH Process API, least-CC mosaic | true-colour RGBA PNG |

Requires CDSE and CMEMS credentials from `.env`.

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path('../.env'))
CDSE_EMAIL    = os.getenv('CDSE_EMAIL')
CDSE_PASSWORD = os.getenv('CDSE_PASSWORD')
assert CDSE_EMAIL and CDSE_PASSWORD, 'CDSE credentials missing from .env'
assert os.getenv('COPERNICUSMARINE_SERVICE_USERNAME'), 'CMEMS credentials missing from .env'

def _get_cdse_token(email, password):
    r = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={'client_id': 'cdse-public', 'username': email,
              'password': password, 'grant_type': 'password'},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()['access_token']

CDSE_TOKEN = _get_cdse_token(CDSE_EMAIL, CDSE_PASSWORD)
_west, _south, _east, _north = AOI_BBOX
print(f'CDSE token:  OK ({len(CDSE_TOKEN)} chars)')
print(f'CMEMS creds: OK')
print(f'AOI bbox:    W={_west} S={_south} E={_east} N={_north}')

In [ ]:
# Open-Meteo wind field: 3×3 sample grid over Baltic AOI → quiver ImageOverlay
_grid_lats = np.linspace(_south + 0.5, _north - 0.5, 3)
_grid_lons = np.linspace(_west  + 0.5, _east  - 0.5, 3)
_LON_G, _LAT_G = np.meshgrid(_grid_lons, _grid_lats)
_U = np.zeros_like(_LON_G)
_V = np.zeros_like(_LON_G)

for _i, _lat in enumerate(_grid_lats):
    for _j, _lon in enumerate(_grid_lons):
        _r = requests.get('https://api.open-meteo.com/v1/forecast', params={
            'latitude': round(float(_lat), 4), 'longitude': round(float(_lon), 4),
            'hourly': 'wind_speed_10m,wind_direction_10m',
            'forecast_days': 1, 'timezone': 'UTC',
        }, timeout=10)
        _h = _r.json()['hourly']
        _spd = _h['wind_speed_10m'][0]
        _deg = _h['wind_direction_10m'][0] % 360
        _rad = np.radians(_deg)
        _U[_i, _j] = -_spd * np.sin(_rad)   # met convention: wind FROM this direction
        _V[_i, _j] = -_spd * np.cos(_rad)

spd_mag = np.sqrt(_U**2 + _V**2)
print(f'Wind grid: {_U.shape} | speed: {spd_mag.min():.1f}–{spd_mag.max():.1f} m/s')

# Render quiver onto a transparent figure whose axes exactly span the AOI bbox
_lon_span = _east - _west
_lat_span = _north - _south
_fig_w = plt.figure(figsize=(8, 8 * _lat_span / _lon_span))
_fig_w.patch.set_alpha(0)
_ax_w = _fig_w.add_axes([0, 0, 1, 1])
_ax_w.patch.set_alpha(0)
_ax_w.set_xlim(_west, _east)
_ax_w.set_ylim(_south, _north)
_ax_w.quiver(_LON_G, _LAT_G, _U, _V, spd_mag,
             cmap='YlOrRd', scale=80, width=0.006,
             headwidth=4, headlength=5, minlength=0, clim=(0, 15))
_ax_w.axis('off')

_buf_wind = io.BytesIO()
_fig_w.savefig(_buf_wind, format='png', dpi=100, transparent=True, bbox_inches=None)
plt.close(_fig_w)
_buf_wind.seek(0)
wind_uri = 'data:image/png;base64,' + base64.b64encode(_buf_wind.getvalue()).decode()
print(f'Wind overlay: {len(wind_uri):,} chars')

In [ ]:
# CMEMS Baltic wave height — colour-mapped ImageOverlay (land/NaN cells transparent)
import copernicusmarine

print('Fetching CMEMS VHM0 subset...')
ds_wave = copernicusmarine.open_dataset(
    dataset_id='cmems_mod_bal_wav_anfc_PT1H-i',
    variables=['VHM0'],
    minimum_longitude=_west,  maximum_longitude=_east,
    minimum_latitude=_south,  maximum_latitude=_north,
)

now_naive = pd.Timestamp(datetime.now(timezone.utc)).tz_localize(None)
t_idx = int(abs(ds_wave.time.values - now_naive.to_datetime64()).argmin())
hs = ds_wave['VHM0'].isel(time=t_idx).values.astype(float)
lats_w = ds_wave.latitude.values
lons_w = ds_wave.longitude.values
t_str = str(ds_wave.time.values[t_idx])[:16]
print(f'Hs shape: {hs.shape} | time: {t_str} | range: {np.nanmin(hs):.2f}–{np.nanmax(hs):.2f} m')

# Colour-map; NaN (land) → alpha=0
norm_wave = plt.Normalize(vmin=0, vmax=2.5)
rgba_wave = plt.get_cmap('Blues')(norm_wave(np.where(np.isnan(hs), 0.0, hs))).copy()
rgba_wave[np.isnan(hs),  3] = 0.0
rgba_wave[~np.isnan(hs), 3] = 0.75

img_wave = Image.fromarray((rgba_wave * 255).astype(np.uint8), 'RGBA')
if lats_w[0] < lats_w[-1]:          # ascending lat → flip so north is at image top
    img_wave = img_wave.transpose(Image.FLIP_TOP_BOTTOM)

buf_wave = io.BytesIO()
img_wave.save(buf_wave, format='PNG')
buf_wave.seek(0)
wave_uri = 'data:image/png;base64,' + base64.b64encode(buf_wave.getvalue()).decode()
print(f'Wave overlay: {len(wave_uri):,} chars')

fig, ax = plt.subplots(figsize=(6, 4))
im_w = ax.imshow(hs, origin='lower',
                 extent=[lons_w.min(), lons_w.max(), lats_w.min(), lats_w.max()],
                 cmap='Blues', vmin=0, vmax=2.5, aspect='auto')
plt.colorbar(im_w, ax=ax, label='Hs (m)')
ax.set_title(f'CMEMS Wave Height — {t_str} UTC')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Sentinel-2 true-colour mosaic via CDSE SH Process API (least cloud cover)
SH_PROCESS_URL = 'https://sh.dataspace.copernicus.eu/api/v1/process'

evalscript = (
    '//VERSION=3\n'
    'function setup() {\n'
    '    return { input: ["B04","B03","B02","dataMask"], output: {bands:4} };\n'
    '}\n'
    'function evaluatePixel(s) {\n'
    '    return [3.5*s.B04, 3.5*s.B03, 3.5*s.B02, s.dataMask];\n'
    '}\n'
)

resp_s2 = requests.post(
    SH_PROCESS_URL,
    json={
        'input': {
            'bounds': {
                'bbox': [_west, _south, _east, _north],
                'properties': {'crs': 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'},
            },
            'data': [{'type': 'sentinel-2-l2a', 'dataFilter': {'mosaickingOrder': 'leastCC'}}],
        },
        'output': {
            'width': 512, 'height': 512,
            'responses': [{'identifier': 'default', 'format': {'type': 'image/png'}}],
        },
        'evalscript': evalscript,
    },
    headers={'Authorization': f'Bearer {CDSE_TOKEN}', 'Content-Type': 'application/json'},
    timeout=60,
)
assert resp_s2.status_code == 200, f'SH API HTTP {resp_s2.status_code}: {resp_s2.text[:200]}'
ct_s2 = resp_s2.headers.get('Content-Type', '')
assert 'image' in ct_s2, f'Expected image, got: {ct_s2!r}'
print(f'Sentinel-2: HTTP 200 | {len(resp_s2.content):,} bytes | {ct_s2}')

s2_uri = 'data:image/png;base64,' + base64.b64encode(resp_s2.content).decode()
print(f'Data URI: {len(s2_uri):,} chars')

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(Image.open(io.BytesIO(resp_s2.content)),
          extent=[_west, _east, _south, _north], aspect='auto')
ax.set_title('Sentinel-2 True Colour — SH Process API (least-CC mosaic)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Full dashboard map: base tiles + WMS context + live data overlays, all toggleable
m_full = Map(
    center=AOI_CENTER,
    zoom=AOI_ZOOM,
    scroll_wheel_zoom=True,
    layout={'height': '650px'},
)

m_full.add(TileLayer(url=OSM_URL, attribution='© OpenStreetMap contributors', name='OpenStreetMap'))
m_full.add(TileLayer(url=SEAMAP_URL, attribution='© OpenSeaMap contributors',
                     name='OpenSeaMap Seamarks', opacity=0.9))
m_full.add(WMSLayer(url=GEBCO_WMS, layers=GEBCO_LAYER, format='image/png',
                    transparent=True, attribution='© GEBCO/BODC',
                    name='GEBCO Bathymetry', opacity=0.4))
m_full.add(WMSLayer(url=EMODNET_WMS, layers=emodnet_layer_active, format='image/png',
                    transparent=True, attribution='© EMODnet Seabed Habitats',
                    name='EMODnet Seabed Habitats', opacity=0.4))
m_full.add(ImageOverlay(url=s2_uri,
                        bounds=[[_south, _west], [_north, _east]],
                        opacity=0.85, name='Sentinel-2 True Colour'))
m_full.add(ImageOverlay(url=wave_uri,
                        bounds=[[_south, _west], [_north, _east]],
                        opacity=0.7, name='Wave Height Hs (m)'))
m_full.add(ImageOverlay(url=wind_uri,
                        bounds=[[_south, _west], [_north, _east]],
                        opacity=0.9, name='Wind Field (m/s)'))
m_full.add(LayersControl(position='topright'))
m_full.add(ScaleControl(position='bottomleft'))
m_full.add(FullScreenControl())
m_full

## 8. Scenario Assertions

In [ ]:
# SCENARIO-MAP-OSM-001
assert resp_osm.status_code == 200
assert 'image' in resp_osm.headers.get('Content-Type', '')
print('PASS  SCENARIO-MAP-OSM-001         OSM tile HTTP 200, image content')

# SCENARIO-MAP-SEAMAP-001
assert resp_sea.status_code == 200
assert 'image' in resp_sea.headers.get('Content-Type', '')
print('PASS  SCENARIO-MAP-SEAMAP-001      OpenSeaMap tile HTTP 200, image content')

# SCENARIO-MAP-GEOTIFF-001
assert depth.shape == (px_h, px_w), f'Unexpected shape: {depth.shape}'
assert 0 <= depth.min() and depth.max() <= 25
assert geotiff_uri.startswith('data:image/png;base64,') and len(geotiff_uri) > 1000
print('PASS  SCENARIO-MAP-GEOTIFF-001     Synthetic raster encoded as PNG data URI ImageOverlay')

# SCENARIO-MAP-WMS-001
print('PASS  SCENARIO-MAP-WMS-001         Generic WMS validated via GEBCO GetMap (HTTP 200)')

# SCENARIO-MAP-GEBCO-001
assert GEBCO_LAYER in gebco_layers, f'{GEBCO_LAYER!r} absent from GetCapabilities'
assert resp_gebco.status_code == 200
assert 'image' in resp_gebco.headers.get('Content-Type', '')
print(f'PASS  SCENARIO-MAP-GEBCO-001       {GEBCO_LAYER!r} confirmed; GetMap HTTP 200')

# SCENARIO-MAP-EMODNET-001
assert len(emodnet_layers) > 0 and emodnet_layer_active is not None
assert resp_emodnet.status_code == 200
assert 'image' in resp_emodnet.headers.get('Content-Type', '')
print(f'PASS  SCENARIO-MAP-EMODNET-001     {emodnet_layer_active!r} confirmed; GetMap HTTP 200')

# SCENARIO-MAP-WIND-OVERLAY-001
assert _U.shape == (3, 3), f'Unexpected wind grid shape: {_U.shape}'
assert wind_uri.startswith('data:image/png;base64,') and len(wind_uri) > 1000
assert spd_mag.shape == (3, 3)
print(f'PASS  SCENARIO-MAP-WIND-OVERLAY-001 Wind quiver overlay: {_U.shape} grid, '
      f'speed {spd_mag.min():.1f}–{spd_mag.max():.1f} m/s')

# SCENARIO-MAP-WAVE-OVERLAY-001
assert not np.all(np.isnan(hs)), 'All-NaN wave height — check CMEMS subset'
assert float(np.nanmax(hs)) <= 10.0, f'Hs out of range: {np.nanmax(hs):.2f} m'
assert wave_uri.startswith('data:image/png;base64,') and len(wave_uri) > 1000
print(f'PASS  SCENARIO-MAP-WAVE-OVERLAY-001 Wave Hs overlay: {hs.shape} grid, '
      f'{np.nanmin(hs):.2f}–{np.nanmax(hs):.2f} m')

# SCENARIO-MAP-S2-OVERLAY-001
assert resp_s2.status_code == 200
assert 'image' in resp_s2.headers.get('Content-Type', '')
assert len(resp_s2.content) > 10_000, f'S2 tile suspiciously small: {len(resp_s2.content)} bytes'
assert s2_uri.startswith('data:image/png;base64,')
print(f'PASS  SCENARIO-MAP-S2-OVERLAY-001   Sentinel-2 overlay: {len(resp_s2.content):,} bytes')

print()
print('=== All SCENARIO-MAP assertions PASS ===')